# Bayesian hyperparameter search — ModernTCN on realized volatilityOptuna's **TPESampler**: Bayesian optimisation with a Tree-structured Parzen Estimator. TPE fitsParzen (kernel-density) estimators to the configurations that scored well and to those that didnot, then proposes the point maximising their ratio — so each trial is chosen from what theprevious ones revealed, rather than drawn at random.Tunes **h = 1** by default. One search per horizon: the best configuration for one horizon is notthe best for another.### What is optimised, and on which rowsEvery trial trains on **2010-01-01 – 2021-12-31** and is scored on **2022-01-01 – 2023-12-31**.The test window (2024-01-01 – 2025-04-07) is **never read during a search** — if it were, thelosses reported afterwards would not be out-of-sample, whatever the final run says.The objective is validation MSE in ln(RV) units by default; `OBJECTIVE` below also offers MAE andQLIKE.### Why the database is not written straight to DriveDrive's FUSE mount does not implement the file locking SQLite needs. A study written directly to`/content/drive/...` will sooner or later fail with *database is locked* or *disk I/O error*, andcan be corrupted rather than merely interrupted.So this notebook keeps the **working database on local disk** and copies a **consistent snapshot**to Drive after every trial, using SQLite's own online-backup API rather than a raw file copy. Ifthe runtime dies you lose the trial in flight and nothing else: re-running the notebook copies thesnapshot back and the search continues. Everything else — the best-parameter JSON, the trialtable, the ready-to-run command — is written straight to Drive, which is safe for ordinary files.The snapshot is a normal Optuna storage, so **optuna-dashboard** opens it either here in Colab(section 6) or on your own machine after downloading it.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# Everything that survives the runtime goes here.
DRIVE_DIR = "/content/drive/MyDrive/ModernTCN_RV"   # <- change if you like
os.makedirs(DRIVE_DIR, exist_ok=True)
print("results will be saved to:", DRIVE_DIR)

## 2. Repository and packages

In [ ]:
import subprocess, sys

REPO   = "https://github.com/Mr0022/ModernTCNt.git"
BRANCH = "claude/moderntcn-aggregation-log-ptj8ao"
ROOT   = "/content/ModernTCNt"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=False)

# os.chdir, not %cd: it moves the Python process, so imports and any ! cells follow.
os.chdir(os.path.join(ROOT, "ModernTCN-Long-term-forecasting"))
print("working directory:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "optuna", "optuna-dashboard"], check=True)

import optuna, torch
print("optuna", optuna.__version__,
      "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## 3. Configuration`N_TRIALS` is how many configurations to try **in this session**. The study is cumulative: run thesearch cell again, or come back tomorrow, and the trials add to what is already there rather thanstarting over.

In [ ]:
HORIZON   = 1            # h: 1 daily, 5 weekly, 22 monthly. One search per horizon.
N_TRIALS  = 100          # configurations to try in THIS session
OBJECTIVE = "mse"        # "mse" | "mae" in ln(RV) units, or "qlike" on the variance scale
EPOCHS    = 50           # cap per trial; early stopping usually ends one sooner
PATIENCE  = 10

STUDY_NAME = f"ModernTCN_h{HORIZON}_{OBJECTIVE}"

# The working database lives on local disk -- SQLite needs locking that Drive's
# FUSE mount does not provide. Only snapshots go to Drive.
LOCAL_DB = f"/content/{STUDY_NAME}.db"
DRIVE_DB = os.path.join(DRIVE_DIR, f"{STUDY_NAME}.db")
STORAGE  = f"sqlite:///{LOCAL_DB}"

# Plain files: JSON, CSV and the re-run command are safe to write to Drive directly.
OUTDIR = os.path.join(DRIVE_DIR, "results_optuna")
os.makedirs(OUTDIR, exist_ok=True)

print(f"study    : {STUDY_NAME}")
print(f"working  : {LOCAL_DB}")
print(f"snapshot : {DRIVE_DB}")
print(f"outputs  : {OUTDIR}")

## 4. Restore any previous runCopies the Drive snapshot back to local disk so the search continues where it stopped. Skipped ona first run, and skipped if a working database is already present.

In [ ]:
import shutil, sqlite3

def snapshot_to_drive(local_db=None, drive_db=None):
    """
    Copy the study to Drive as a consistent snapshot.

    sqlite3's online-backup API is used rather than a file copy: the search holds
    the database open, and a raw copy taken mid-write can land on Drive
    half-updated. The backup is taken to a LOCAL temporary file and only then
    byte-copied across, because writing a SQLite file directly onto the FUSE
    mount is the very thing that is unreliable.
    """
    local_db = local_db or LOCAL_DB
    drive_db = drive_db or DRIVE_DB
    if not os.path.exists(local_db):
        return False
    tmp = local_db + ".snapshot"
    try:
        if os.path.exists(tmp):
            os.remove(tmp)
        src = sqlite3.connect(local_db)
        dst = sqlite3.connect(tmp)
        with dst:
            src.backup(dst)
        src.close(); dst.close()
        shutil.copy2(tmp, drive_db)
        return True
    except Exception as e:
        print(f"  [drive] snapshot failed: {e}")
        return False
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


if os.path.exists(LOCAL_DB):
    print("A working database is already present; leaving it alone.")
elif os.path.exists(DRIVE_DB):
    shutil.copy2(DRIVE_DB, LOCAL_DB)
    import optuna as _o
    n = len(_o.load_study(study_name=STUDY_NAME, storage=STORAGE).trials)
    print(f"Restored from Drive: {n} trial(s) already recorded. The search will continue.")
else:
    print("No previous study found. Starting a new one.")

## 5. Run the searchEach finished trial triggers a Drive snapshot, so the cost of a dropped runtime is one trial.Expect a few seconds to a couple of minutes per trial depending on the configuration drawn andwhether you are on a GPU — `Runtime → Change runtime type → GPU` is worth setting.Interrupting this cell is safe: completed trials are already in the database and on Drive.

In [ ]:
import time
import tune_optuna

started = time.time()
saved   = {"n": 0}

def on_trial_end(study, trial):
    """Snapshot after every trial and print a running best."""
    if snapshot_to_drive():
        saved["n"] += 1
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if done:
        print(f"  [{len(study.trials):3d} trials | {len(done)} complete | "
              f"best {study.best_value:.6f} | {(time.time()-started)/60:.1f} min]")

argv = [
    "--pred_len",      str(HORIZON),
    "--n_trials",      str(N_TRIALS),
    "--objective",     OBJECTIVE,
    "--train_epochs",  str(EPOCHS),
    "--patience",      str(PATIENCE),
    "--storage",       STORAGE,
    "--study_name",    STUDY_NAME,
    "--outdir",        OUTDIR,
    "--num_workers",   "2",
]

study = tune_optuna.main(argv, callbacks=[on_trial_end])

snapshot_to_drive()
print(f"\nDrive snapshots written: {saved['n']}")
print(f"Study database: {DRIVE_DB}")

## 6. Results`tune_optuna.main` has already written `optuna_h{h}_best.json`, `_trials.csv` and `_command.sh`into the Drive output folder. The table below is the same data, sorted best first.

In [ ]:
import pandas as pd
from IPython.display import display

df = study.trials_dataframe()
complete = df[df.state == "COMPLETE"].sort_values("value")
unit = "RV" if OBJECTIVE == "qlike" else "ln"

print(f"{len(complete)} complete, {(df.state == 'PRUNED').sum()} pruned, "
      f"{(df.state == 'FAIL').sum()} failed")
print(f"best validation {OBJECTIVE.upper()}[{unit}] = {study.best_value:.6f} "
      f"(trial {study.best_trial.number})\n")

cols = ["number", "value", "user_attrs_n_params"] + \
       [c for c in complete.columns if c.startswith("params_")]
display(complete[cols].head(15).reset_index(drop=True))

print("\nBest configuration:")
for k, v in study.best_trial.params.items():
    print(f"  {k:<16} {v}")

## 7. Optuna DashboardThe snapshot is an ordinary Optuna storage, so the dashboard reads it unchanged.**In Colab** — the cell below serves the dashboard on a port and opens it in a new browser tab.If the tab is blocked, allow pop-ups for `colab.research.google.com` and re-run it.

In [ ]:
import threading
from optuna_dashboard import run_server

PORT = 8080

# run_server blocks, so it goes on a daemon thread; the kernel stays responsive
# and the thread dies with the runtime.
threading.Thread(target=run_server,
                 kwargs={"storage": STORAGE, "host": "localhost", "port": PORT},
                 daemon=True).start()
time.sleep(3)   # let the server bind before the proxy is asked for it

try:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT)
    print(f"Dashboard serving on port {PORT} — opening a new tab.")
except Exception as e:
    print(f"Could not open the Colab proxy ({e}).")
    print("The dashboard is still running; see the local instructions below.")

**On your own machine** — download `<STUDY_NAME>.db` from your Drive folder, then:```bashpip install optuna-dashboardoptuna-dashboard sqlite:///ModernTCN_h1_mse.db```and open <http://127.0.0.1:8080>. This is the better option for reading the results properly:history, parallel-coordinate and slice plots, hyperparameter importances, and the full trial table,all interactive.

## 8. Re-run the winner over 5 seedsThe search trains each trial under a single seed to stay affordable, so the best value above is onedraw. This trains the winning configuration over seeds 2021–2025 and reports the mean and standarddeviation on the **test** window — the first and only time the test rows are read.

In [ ]:
cmd_file = os.path.join(OUTDIR, f"optuna_h{HORIZON}_command.sh")
cmd = open(cmd_file).read().strip()
print(cmd + "\n")

# Uncomment to run it (a few minutes):
# !{cmd}

## 9. What is on Drive| path | what ||---|---|| `ModernTCN_h{h}_{objective}.db` | the study — open with optuna-dashboard, resumable || `results_optuna/optuna_h{h}_best.json` | best value and parameters || `results_optuna/optuna_h{h}_trials.csv` | every trial, its parameters and its model size || `results_optuna/optuna_h{h}_command.sh` | the `run.py` invocation for the winner |To add more trials later, just re-run this notebook: section 4 restores the study and section 5continues it. Raise `N_TRIALS` for a longer session, or change `HORIZON` to search h = 5 or 22 —each horizon gets its own study and its own files.